In [16]:
 
import os
import logging
from pathlib import Path
 
from langchain_community.document_loaders import JSONLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

In [17]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
 
CONFIG = {
    "json_path": "/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json",
    "embedding_model": "/Users/muhammadzuamaalamin/Documents/fintunellm/model/bge-m3",
    "faiss_index_path": "faiss_index",
    "llm_model": "gemma3:4b",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "bm25_k": 5,
    "vector_k": 3,
    "llm_temperature": 0.1,
}


In [18]:
 
PROMPT_TEMPLATE = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Kamu adalah asisten ahli hukum tata negara Indonesia yang memahami Undang-Undang Dasar 1945.
 
Jawablah pertanyaan berikut HANYA berdasarkan konteks dokumen yang diberikan.
Berikan jawaban yang jelas, akurat, dan mudah dipahami.
Jika relevan, sebutkan pasal dan ayat yang menjadi dasar jawabanmu.
 
Pertanyaan:
{question}
 
Konteks:
{context}
 
Panduan menjawab:
- Jawab berdasarkan isi dokumen saja
- Sebutkan "Pasal X Ayat Y" jika mengutip pasal tertentu
- Jangan menambahkan informasi di luar konteks dokumen
 
Jika jawaban tidak ditemukan dalam dokumen, katakan:
"Maaf, informasi tersebut tidak dijelaskan secara spesifik dalam Undang-Undang Dasar 1945."
""",
)

In [19]:
 
def load_documents(json_path: str) -> list:
    """Load dokumen dari file JSON."""
    logger.info("Memuat dokumen dari: %s", json_path)
    loader = JSONLoader(
        file_path=json_path,
        jq_schema=".[]",
        text_content=False,
    )
    docs = loader.load()
    logger.info("Berhasil memuat %d dokumen", len(docs))
    return docs
 
 
def split_documents(docs: list, chunk_size: int = 500, chunk_overlap: int = 50) -> list:
    """Split dokumen menjadi chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    logger.info("Menghasilkan %d chunks dari %d dokumen", len(chunks), len(docs))
    return chunks
 

In [20]:
 
def load_or_create_vectorstore(chunks: list, cfg: dict) -> FAISS:
    """Load FAISS index jika ada, atau buat baru."""
    embeddings = HuggingFaceEmbeddings(
        model_name=cfg["embedding_model"],
        model_kwargs={"device": "cpu"},
    )
 
    faiss_path = cfg["faiss_index_path"]
 
    if Path(faiss_path).exists():
        logger.info("Memuat FAISS index dari: %s", faiss_path)
        vectorstore = FAISS.load_local(
            faiss_path,
            embeddings,
            allow_dangerous_deserialization=True,
        )
    else:
        logger.info("Membuat FAISS index baru...")
        vectorstore = FAISS.from_documents(chunks, embeddings)
        vectorstore.save_local(faiss_path)
        logger.info("FAISS index disimpan ke: %s", faiss_path)
 
    return vectorstore

In [21]:

def build_hybrid_retriever(chunks: list, vectorstore: FAISS, cfg: dict) -> EnsembleRetriever:
    """Buat hybrid retriever (BM25 + FAISS)."""
    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = cfg["bm25_k"]
 
    vector_retriever = vectorstore.as_retriever(
        search_kwargs={"k": cfg["vector_k"]}
    )
 
    hybrid = EnsembleRetriever(
        retrievers=[bm25, vector_retriever],
        weights=[0.5, 0.5],
    )
    logger.info(
        "Hybrid retriever siap (BM25 k=%d, vector k=%d)",
        cfg["bm25_k"],
        cfg["vector_k"],
    )
    return hybrid
 
# ---------------------------------------------------------------------------
# Fase 4: QA chain
# ---------------------------------------------------------------------------
 
def build_qa_chain(retriever: EnsembleRetriever, cfg: dict) -> RetrievalQA:
    """Buat RetrievalQA chain."""
    llm = ChatOllama(model=cfg["llm_model"], temperature=cfg["llm_temperature"])
 
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": PROMPT_TEMPLATE},
        return_source_documents=True,
    )
    logger.info("QA chain siap dengan model: %s", cfg["llm_model"])
    return qa_chain
 
# ---------------------------------------------------------------------------
# Fase 5: Query helper
# ---------------------------------------------------------------------------
 
def ask(qa_chain: RetrievalQA, question: str, show_sources: bool = False) -> str:
    """Kirim pertanyaan ke QA chain dan kembalikan jawaban."""
    logger.info("Pertanyaan: %s", question)
 
    # PERBAIKAN: gunakan .invoke() bukan .run() / manual call
    response = qa_chain.invoke({"query": question})
 
    answer = response["result"]
 
    if show_sources:
        sources = response.get("source_documents", [])
        print("\n--- Dokumen sumber ---")
        for i, doc in enumerate(sources, 1):
            print(f"\n[{i}] {doc.page_content[:300]}")
        print("----------------------\n")
 
    return answer
 
 
def demo_similarity_search(vectorstore: FAISS, query: str, k: int = 5):
    """Demo: tampilkan hasil similarity search dengan skor."""
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f"\n--- Similarity search: '{query}' ---")
    for i, (doc, score) in enumerate(results, 1):
        print(f"\nResult {i} | score={score:.4f}")
        print(doc.page_content[:400])
    print("------------------------------------\n")
 
 
def demo_hybrid_retrieve(retriever: EnsembleRetriever, query: str):
    """Demo: tampilkan hasil hybrid retrieval.
    
    PERBAIKAN: pakai .invoke() bukan .get_relevant_documents() yang deprecated,
    dan iterasi docs hasil retrieval (bukan variable 'results' lama).
    """
    # PERBAIKAN: gunakan .invoke() — get_relevant_documents() deprecated
    docs = retriever.invoke(query)
    print(f"\n--- Hybrid retrieval: '{query}' ---")
    for i, doc in enumerate(docs, 1):  # PERBAIKAN: iterasi 'docs', bukan 'results'
        print(f"\n[{i}] {doc.page_content[:400]}")
    print("-----------------------------------\n")
 

In [22]:
def main():
    try:
        # Load & proses dokumen
        docs = load_documents(CONFIG["json_path"])
        chunks = split_documents(docs, CONFIG["chunk_size"], CONFIG["chunk_overlap"])
 
        # Build index & retriever
        vectorstore = load_or_create_vectorstore(chunks, CONFIG)
        hybrid_retriever = build_hybrid_retriever(chunks, vectorstore, CONFIG)
 
        # Demo retrieval (opsional, bisa dihapus di produksi)
        demo_similarity_search(vectorstore, "bunyi pasal 1 uud 1945")
        demo_hybrid_retrieve(hybrid_retriever, "Apa hak-hak warga negara yang dijamin dalam UUD 1945?")
 
        # Build QA chain
        qa_chain = build_qa_chain(hybrid_retriever, CONFIG)
 
        # Contoh query
        questions = [
            "Bunyi pasal 1 UUD 1945?",
            "Apa hak-hak warga negara yang dijamin dalam UUD 1945?",
            "Bagaimana sistem pemerintahan Indonesia menurut UUD 1945?",
        ]
 
        for question in questions:
            print(f"\n{'='*60}")
            print(f"Pertanyaan: {question}")
            print(f"{'='*60}")
            answer = ask(qa_chain, question, show_sources=True)
            print(f"Jawaban:\n{answer}")
 
    except FileNotFoundError as e:
        logger.error("File tidak ditemukan: %s", e)
    except Exception as e:
        logger.exception("Terjadi error: %s", e)
        raise
 
 
if __name__ == "__main__":
    main()

2026-03-16 10:40:05,340 [INFO] Memuat dokumen dari: /Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json
2026-03-16 10:40:05,387 [INFO] Berhasil memuat 71 dokumen
2026-03-16 10:40:05,388 [INFO] Menghasilkan 73 chunks dari 71 dokumen
2026-03-16 10:40:05,392 [INFO] Load pretrained SentenceTransformer: /Users/muhammadzuamaalamin/Documents/fintunellm/model/bge-m3
2026-03-16 10:40:07,198 [INFO] Memuat FAISS index dari: faiss_index
2026-03-16 10:40:07,208 [INFO] Hybrid retriever siap (BM25 k=5, vector k=3)



--- Similarity search: 'bunyi pasal 1 uud 1945' ---

Result 1 | score=1.0627
{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}

Result 2 | score=1.0989
{"id": "psl_3_ay_1", "pasal": 3, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat menetapkan Undang-Undang Dasar dan garis-garis besar dari ada haluan negara."}

Result 3 | score=1.1279
{"id": "psl_ap_2_ay_1", "pasal": 2, "ayat": 1, "bab": "ATURAN PERALIHAN", "content": "Segala badan negara dan peraturan yang ada masih langsung berlaku, selama belum diadakan yang baru menurut Undang-Undang Dasar ini."}

Result 4 | score=1.1286
{"id": "psl_1_ay_2", "pasal": 1, "ayat": 2, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat."}

Result 5 | score=1.1386
{"id": "psl_15_ay_1", "pasal

2026-03-16 10:40:07,806 [INFO] QA chain siap dengan model: gemma3:4b
2026-03-16 10:40:07,807 [INFO] Pertanyaan: Bunyi pasal 1 UUD 1945?



--- Hybrid retrieval: 'Apa hak-hak warga negara yang dijamin dalam UUD 1945?' ---

[1] {"id": "psl_27_ay_2", "pasal": 27, "ayat": 2, "bab": "BAB X - WARGA NEGARA", "content": "Tiap-tiap warga negara berhak atas pekerjaan dan penghidupan yang layak bagi kemanusiaan."}

[2] {"id": "psl_27_ay_1", "pasal": 27, "ayat": 1, "bab": "BAB X - WARGA NEGARA", "content": "Segala warga negara bersamaan kedudukannya dalam hukum dan pemerintahan dan wajib menjunjung hukum dan pemerintahan itu dengan tidak ada kecualinya."}

[3] {"id": "psl_30_ay_1", "pasal": 30, "ayat": 1, "bab": "BAB XII - PERTAHANAN NEGARA", "content": "Tiap-tiap warga negara berhak dan wajib ikut serta dalam usaha pembelaan negara."}

[4] {"id": "psl_18_ay_1", "pasal": 18, "ayat": 1, "bab": "BAB VI - PEMERINTAHAN DAERAH", "content": "Pembagian daerah Indonesia atas daerah besar dan kecil, dengan bentuk susunan pemerintahannya ditetapkan dengan undang-undang, dengan memandang dan mengingati dasar permusyawaratan dalam sistem pemeri

2026-03-16 10:40:12,673 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-03-16 10:40:13,916 [INFO] Pertanyaan: Apa hak-hak warga negara yang dijamin dalam UUD 1945?



--- Dokumen sumber ---

[1] {"id": "psl_at_1_ay_2", "pasal": 1, "ayat": 2, "bab": "ATURAN PERTAMBAHAN", "content": "Dalam enam bulan sesudah Majelis Permusyawaratan Rakyat dibentuk, Majelis itu bersidang untuk menetapkan Undang-Undang Dasar."}

[2] {"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}

[3] {"id": "psl_21_ay_2", "pasal": 21, "ayat": 2, "bab": "BAB VII - DEWAN PERWAKILAN RAKYAT", "content": "Jika rancangan itu, meskipun disetujui oleh Dewan Perwakilan Rakyat, tidak disyahkan oleh Presiden, maka rancangan tadi tidak boleh dimajukan lagi dalam persidangan Dewan Perwakilan Rakyat masa itu.

[4] {"id": "psl_3_ay_1", "pasal": 3, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat menetapkan Undang-Undang Dasar dan garis-garis besar dari ada haluan negara."}

[5] {"id": "psl_12_ay_1", "pasal": 12, "ayat": 1, "bab": "BA

2026-03-16 10:40:15,454 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-03-16 10:40:22,308 [INFO] Pertanyaan: Bagaimana sistem pemerintahan Indonesia menurut UUD 1945?



--- Dokumen sumber ---

[1] {"id": "psl_27_ay_2", "pasal": 27, "ayat": 2, "bab": "BAB X - WARGA NEGARA", "content": "Tiap-tiap warga negara berhak atas pekerjaan dan penghidupan yang layak bagi kemanusiaan."}

[2] {"id": "psl_27_ay_1", "pasal": 27, "ayat": 1, "bab": "BAB X - WARGA NEGARA", "content": "Segala warga negara bersamaan kedudukannya dalam hukum dan pemerintahan dan wajib menjunjung hukum dan pemerintahan itu dengan tidak ada kecualinya."}

[3] {"id": "psl_30_ay_1", "pasal": 30, "ayat": 1, "bab": "BAB XII - PERTAHANAN NEGARA", "content": "Tiap-tiap warga negara berhak dan wajib ikut serta dalam usaha pembelaan negara."}

[4] {"id": "psl_18_ay_1", "pasal": 18, "ayat": 1, "bab": "BAB VI - PEMERINTAHAN DAERAH", "content": "Pembagian daerah Indonesia atas daerah besar dan kecil, dengan bentuk susunan pemerintahannya ditetapkan dengan undang-undang, dengan memandang dan mengingati dasar permusyawaratan dalam sistem pemerinta

[5] {"id": "psl_26_ay_1", "pasal": 26, "ayat": 1, "bab

2026-03-16 10:40:23,819 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



--- Dokumen sumber ---

[1] {"id": "psl_4_ay_1", "pasal": 4, "ayat": 1, "bab": "BAB III - KEKUASAAN PEMERINTAH NEGARA", "content": "Presiden Republik Indonesia memegang kekuasaan pemerintahan menurut Undang-Undang Dasar."}

[2] {"id": "psl_18_ay_1", "pasal": 18, "ayat": 1, "bab": "BAB VI - PEMERINTAHAN DAERAH", "content": "Pembagian daerah Indonesia atas daerah besar dan kecil, dengan bentuk susunan pemerintahannya ditetapkan dengan undang-undang, dengan memandang dan mengingati dasar permusyawaratan dalam sistem pemerinta

[3] Republik Indonesia) dengan sebaik-baiknya dan seadil-adilnya, memegang teguh Undang- Undang Dasar dan menjalankan segala undang-undang dan peraturannya dengan selurus-lurusnya serta berbakti kepada Nusa dan Bangsa

[4] {"id": "psl_ap_1_ay_1", "pasal": 1, "ayat": 1, "bab": "ATURAN PERALIHAN", "content": "Panitia Persiapan Kemerdekaan Indonesia mengatur dan menyelenggarakan kepindahan pemerintahan kepada Pemerintah Indonesia."}

[5] {"id": "psl_27_ay_1", "pasal":